In [ ]:
# =============================================================================
# PIML Evaluation Suite — Lean Protocol (v1.0)
#
# Purpose:
#   Rank and evaluate PIML experiments using a minimal, defensible, multi-metric
#   scoring rule prioritizing Test > Validation and Cumulative > Aggregate.
#
# Philosophy:
#   - Minimal viable statistics (ANOVA+Tukey only on primary metric, optional)
#   - Rank-based, per-well scoring → global ranking (average across wells)
#   - Simple, clear plots and an executive summary
#
# Usage:
#   - Configure REPORTS_DIR and EXPERIMENTS below.
#   - Run as a script or import PIMLEvaluationSuite and call .run()
# =============================================================================

from __future__ import annotations
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Optional stats (ANOVA + Tukey)
try:
    import statsmodels.api as sm  # noqa: F401
    from statsmodels.formula.api import ols
    from statsmodels.stats.anova import anova_lm
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    STATSMODELS_AVAILABLE = True
except Exception:
    STATSMODELS_AVAILABLE = False
    warnings.warn(
        "statsmodels not found. Statistical analysis (ANOVA, Tukey) will be skipped.\n"
        "Install with: pip install statsmodels"
    )

# ------------------------------------------------------------
# Project root (optional fallback for notebooks/scripts)
# ------------------------------------------------------------
try:
    project_root = Path(get_ipython().run_line_magic('pwd')[0].split('/notebooks')[0])
except Exception:
    project_root = Path.cwd().parent.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# =============================================================================
# Configuration (data, evaluation, visuals)
# =============================================================================

# =============================================================================
# PIML Evaluation Suite — Protocol v1.1 (Hybrid: 50% Rank + 50% Effect)
#
# What changed vs v1.0
# - Per-well, per-metric score = 0.5 * normalized-rank  +  0.5 * normalized-effect
#   • normalized-rank: (rank-1)/(K-1)   (0 = best, 1 = worst)
#   • normalized-effect: winsorized min–max per well/metric (0 = best, 1 = worst)
# - Relative overfitting rule: flag & soft-penalize when
#     (test - val) / max(val, eps) >= tau  OR  (test - val) >= abs_thr
#   (defaults: tau=25%, eps=0.5 pp, abs_thr=2 pp, lambda=0.5)
# - Weights (test/val, cum/agg) remain configurable.
# =============================================================================

# ---------------------------------------------------------------------
# You said imports & project_root are already defined in your notebook.
# Only the data-paths and config below matter for this script.
# ---------------------------------------------------------------------

EXP = "WARM_1"

# ---- Data (adjust) ----
REPORTS_DIR = project_root / "src" / "experiment_configs" / EXP / "reports" / "seq2"
EXPERIMENTS = [
    "reconstruct", "hp_hist", "hp_raw", "hp_hist_warm", "hp_raw_warm",
    "reconstruct_warm_raw", "reconstruct_warm_hp", "reconstruct_warm_ewma",
    "reconstruct_warm_holt",
]
CSV_STEM_PREFIX = "validation_"  # e.g., validation_<experiment>.csv

# ---- Evaluation protocol ----
@dataclass
class EvaluationConfig:
    protocol_version: str = "1.1"
    # Core metrics (must exist or be creatable as NaN)
    metrics: List[str] = field(default_factory=lambda: [
        "val_smape_agg", "val_smape_cum", "test_smape_agg", "test_smape_cum"
    ])
    # Primary business metric for optional stats
    primary_metric: str = "test_smape_cum"
    # Weighted score: (note: lower is better throughout)
    # These are *combination* weights after per-metric hybrid scoring.
    weights: Dict[str, float] = field(default_factory=lambda: {
        "test": 0.4, "val": 0.6, "cum": 0.4, "agg": 0.6
    })
    # Relative overfitting rule (applies to agg and cum independently)
    overfit_rel_tau: float = 0.25    # 25%
    overfit_eps: float = 0.5         # pp guard for tiny val
    overfit_abs_thr: float = 3.0     # pp absolute fallback
    overfit_lambda: float = 0.5      # soft penalty slope on excess
    # Policy: drop rows missing test metrics
    require_test_metrics: bool = True
    # Winsorization for effect normalization (per well/metric)
    winsor_pcts: Tuple[float, float] = (5.0, 95.0)

    overfit_apply: Dict[str, bool] = field(default_factory=lambda: {
        "agg": True,   # apply to aggregate
        "cum": False,  # ignore cumulative (your request)
    })

# ---- Visual tokens (kept from your scheme) ----
@dataclass
class StyleConfig:
    font_family: str = "Inter, Arial, sans-serif"
    color_text: str = "#2c3e50"
    color_grid: str = "#E6E6E6"
    UI_COLORS: Dict[str, str] = field(default_factory=lambda: {
        "text": "#222222",
        "grid": "#E6E6E6",
    })
    SEMANTIC_COLORS: Dict[str, str] = field(default_factory=lambda: {
        "validation": "#2ecc71",
        "test": "#27ae60",
        "test_accent": "#7fbf90",
        "train_primary": "#4f6bd8",
        "train_secondary": "#2c3e50",
    })
    PALETTE: Dict[str, str] = field(default_factory=lambda: {
        "blue_vivid": '#0077B6',
        "green_lime": '#32CD32',
        "green_muted": '#84a98c',
        "blue_dark_text": '#2c3e50',
        "gray_neutral": '#7f7f7f',
    })
    experiment_colors: Dict[str, str] = field(default_factory=lambda: {
        "reconstruct": '#0077B6',
        "hp_hist": '#4f6bd8',
        "hp_raw": '#2c3e50',
        "hp_hist_warm": '#2ecc71',
        "hp_raw_warm": '#27ae60',
        "reconstruct_warm_raw": '#32CD32',
        "reconstruct_warm_hp": '#84a98c',
        "reconstruct_warm_ewma": '#2c3e50',
        "reconstruct_warm_holt": '#7f7f7f',
    })

# =============================================================================
# Core Suite
# =============================================================================

class PIMLEvaluationSuite:
    """
    Lean evaluation pipeline (Hybrid Protocol v1.1):
      1) Load & normalize CSVs
      2) Enforce policy (drop runs without test metrics)
      3) Collapse duplicates per (well, experiment) via mean
      4) For each metric & well: compute
           - normalized-rank (0 best .. 1 worst)
           - normalized-effect via winsorized min–max (0 best .. 1 worst)
           - hybrid score = 0.5*rank + 0.5*effect
      5) Combine metrics with weights into per-well composite score
      6) Apply soft overfitting penalty (relative rule)
      7) Rank globally (avg score across wells)
      8) Optional ANOVA/Tukey on primary metric
      9) Plots + executive summary
    """
    def __init__(
        self,
        reports_dir: Path,
        experiments: List[str],
        csv_prefix: str,
        eval_config: EvaluationConfig,
        style_config: StyleConfig,
    ):
        self.reports_dir = Path(reports_dir)
        self.experiments = experiments
        self.csv_prefix = csv_prefix
        self.cfg = eval_config
        self.style = style_config

        # State
        self.proc_df: Optional[pd.DataFrame] = None
        self.global_ranking: Optional[pd.DataFrame] = None
        self.per_well_winners: Optional[pd.DataFrame] = None
        self.anova_table: Optional[pd.DataFrame] = None
        self.tukey_letters: Dict[str, str] = {}

    # -------------------------------
    # Public entry point
    # -------------------------------
    def run(self) -> None:
        print(f"--- PIML Evaluation Suite (Protocol v{self.cfg.protocol_version}) — Hybrid 50/50 ---")
        self._load_and_preprocess()
        self._score_and_rank_hybrid()
        if STATSMODELS_AVAILABLE:
            self._stats_on_primary_metric()
        self._render_outputs()
        print("--- Evaluation complete ---")

    # -------------------------------
    # Phase 1 — Load & Preprocess
    # -------------------------------
    def _load_and_preprocess(self) -> None:
        print("1) Loading & preprocessing…")
        paths = self._find_csvs()
        if not paths:
            raise FileNotFoundError(f"No CSVs found under {self.reports_dir}")

        frames: List[pd.DataFrame] = []
        for exp, p in paths.items():
            try:
                df = pd.read_csv(p)
                if df is None or df.empty:
                    continue
                df = df.copy()
                df["experiment"] = exp
                frames.append(df)
            except Exception as e:
                print(f"[warn] could not read {p}: {e}")

        if not frames:
            raise ValueError("All CSVs failed to load or were empty.")

        df = pd.concat(frames, ignore_index=True)

        # Normalize naming
        if "architecture" not in df.columns and "architecture_name" in df.columns:
            df = df.rename(columns={"architecture_name": "architecture"})
        df["well"] = df["well"].astype(str)
        df["experiment"] = df["experiment"].astype(str)

        # Ensure metric columns
        for m in self.cfg.metrics:
            if m not in df.columns:
                df[m] = np.nan
            df[m] = pd.to_numeric(df[m], errors="coerce")

        # Policy: require test metrics
        pre_n = len(df)
        if self.cfg.require_test_metrics:
            df = df.dropna(subset=["test_smape_agg", "test_smape_cum"])
        print(f"   - dropped {pre_n - len(df)} rows (missing test metrics)")

        # Remove inf / coerce
        df = df.replace([np.inf, -np.inf], np.nan)

        # Collapse duplicates per (well, experiment) by mean (avoids pseudo-replication)
        group_cols = ["well", "experiment"]
        df = df.groupby(group_cols, as_index=False)[self.cfg.metrics].mean()

        self.proc_df = df

    # -------------------------------
    # Phase 2 — Hybrid Score & Rank
    # -------------------------------
    def _score_and_rank_hybrid(self) -> None:
        print("2) Ranking & hybrid scoring…")
        assert self.proc_df is not None
        df = self.proc_df.copy()
    
        # ---- Step 2.1: normalized ranks per well & metric (lower is better)
        for m in self.cfg.metrics:
            r = df.groupby("well")[m].rank(method="min", ascending=True)
            k = df.groupby("well")[m].transform("count")  # number of contenders in that well
            denom = (k - 1).replace(0, np.nan)           # if only one contender, denom=nan
            df[f"rank_{m}"] = ((r - 1) / denom).fillna(0.0)  # 0 best .. 1 worst
    
        # ---- Step 2.2: normalized effects via winsorized min–max (robust)
        p_lo, p_hi = self.cfg.winsor_pcts
        p_lo_f, p_hi_f = p_lo / 100.0, p_hi / 100.0
    
        for m in self.cfg.metrics:
            # compute per-well quantiles with explicit names, then merge
            q = (
                df.groupby("well")[m]
                  .quantile([p_lo_f, p_hi_f])
                  .unstack(level=1)
                  .rename(columns={p_lo_f: f"lo_{m}", p_hi_f: f"hi_{m}"})
                  .reset_index()
            )
            df = df.merge(q, on="well", how="left")
    
            # clamp and scale to [0,1]
            lo = df[f"lo_{m}"].to_numpy()
            hi = df[f"hi_{m}"].to_numpy()
            x  = df[m].to_numpy()
    
            # if hi==lo or NaN, fall back to 0.5 (uninformative)
            hi_safe = hi.copy()
            hi_safe[np.isnan(hi_safe)] = lo[np.isnan(hi_safe)]
            denom = hi_safe - lo
            denom[denom == 0] = np.nan
    
            x_clamped = np.minimum(np.maximum(x, lo), hi_safe)
            eff = (x_clamped - lo) / denom
            df[f"effect_{m}"] = pd.Series(eff, index=df.index).fillna(0.5)
    
            # cleanup helpers for this metric
            df.drop(columns=[f"lo_{m}", f"hi_{m}"], inplace=True, errors="ignore")
    
        # ---- Step 2.3: hybrid per-metric score (0 best .. 1 worst)
        for m in self.cfg.metrics:
            df[f"hybrid_{m}"] = 0.5 * df[f"rank_{m}"] + 0.5 * df[f"effect_{m}"]
    
        # ---- Step 2.4: combine metrics into per-well composite (lower = better)
        w = self.cfg.weights
        df["score_val"]  = w["agg"] * df["hybrid_val_smape_agg"]  + w["cum"] * df["hybrid_val_smape_cum"]
        df["score_test"] = w["agg"] * df["hybrid_test_smape_agg"] + w["cum"] * df["hybrid_test_smape_cum"]
        df["score_well_raw"] = w["val"] * df["score_val"] + w["test"] * df["score_test"]
    
        # ---- Step 2.5: relative overfitting (soft penalty) ----
        use_agg = bool(self.cfg.overfit_apply.get("agg", True))
        use_cum = bool(self.cfg.overfit_apply.get("cum", False))
        
        def _overfit_components(row, m_test, m_val, rel_tau, abs_thr, eps):
            test = float(row[m_test]); val = float(row[m_val])
            d_abs = test - val
            rel = d_abs / max(val, eps)
            triggered = (rel >= rel_tau) or (d_abs >= abs_thr)
            excess = max(0.0, rel - rel_tau)  # only “excess” over the relative threshold feeds penalty
            return triggered, rel, d_abs, excess
        
        trig_agg, rel_agg, dagg, exc_agg = [], [], [], []
        trig_cum, rel_cum, dcum, exc_cum = [], [], [], []
        
        for _, row in df.iterrows():
            # agg
            ta, ra, da, ea = _overfit_components(
                row, "test_smape_agg", "val_smape_agg",
                self.cfg.overfit_rel_tau, self.cfg.overfit_abs_thr, self.cfg.overfit_eps
            )
            # cum
            tc, rc, dc, ec = _overfit_components(
                row, "test_smape_cum", "val_smape_cum",
                self.cfg.overfit_rel_tau, self.cfg.overfit_abs_thr, self.cfg.overfit_eps
            )
            trig_agg.append(ta); rel_agg.append(ra); dagg.append(da); exc_agg.append(ea)
            trig_cum.append(tc); rel_cum.append(rc); dcum.append(dc); exc_cum.append(ec)
        
        # Flags (sempre calculamos para inspecionar), mas a penalização pode ignorar cum
        df["overfit_agg_flag"] = trig_agg
        df["overfit_cum_flag"] = trig_cum
        df["overfit_flag"] = (df["overfit_agg_flag"] & use_agg) | (df["overfit_cum_flag"] & use_cum)
        df["overfit_rel_agg"] = rel_agg
        df["overfit_rel_cum"] = rel_cum
        df["overfit_abs_agg"] = dagg
        df["overfit_abs_cum"] = dcum
        
        # Apenas o(s) eixo(s) habilitado(s) alimenta(m) penalty
        exc_agg_arr = np.array(exc_agg) if use_agg else np.zeros(len(df))
        exc_cum_arr = np.array(exc_cum) if use_cum else np.zeros(len(df))
        excess = np.maximum(exc_agg_arr, exc_cum_arr)
        
        penalty = 1.0 + self.cfg.overfit_lambda * excess
        df["score_well"] = df["score_well_raw"] * penalty

    
        # ---- Step 2.6: global ranking & per-well winners
        global_rank = (
            df.groupby("experiment")["score_well"]
              .agg(mean_score="mean", std_score="std", sem_score=lambda s: s.sem())
              .sort_values("mean_score", ascending=True)
              .reset_index()
        )
        idx = df.groupby("well")["score_well"].idxmin()
        winners = df.loc[idx].sort_values("well").reset_index(drop=True)
    
        self.proc_df = df
        self.global_ranking = global_rank
        self.per_well_winners = winners


    # -------------------------------
    # Phase 3 — Minimal Stats (Optional)
    # -------------------------------
    def _stats_on_primary_metric(self) -> None:
        print("3) Stats (primary metric) …")
        assert self.proc_df is not None
        m = self.cfg.primary_metric
        df = self.proc_df.dropna(subset=[m])
        if df.empty or df["experiment"].nunique() < 2:
            print("   - not enough data/groups; skipping stats.")
            return
        try:
            # Two-way ANOVA: primary_metric ~ C(experiment) + C(well)
            model = ols(f"`{m}` ~ C(experiment) + C(well)", data=df).fit()
            self.anova_table = anova_lm(model, typ=2)
            # Tukey HSD on experiment groups (per primary metric values)
            tukey = pairwise_tukeyhsd(endog=df[m], groups=df["experiment"], alpha=0.05)
            self.tukey_letters = self._compact_letter_display_from_tukey(tukey)
        except Exception as e:
            print(f"[warn] stats failed: {e}")

    # -------------------------------
    # Phase 4 — Outputs
    # -------------------------------
    def _render_outputs(self) -> None:
        print("4) Outputs …")

        # Tables
        display(HTML("<h3>Table 1 — Overall Model Ranking</h3><p>Mean composite score across wells (lower is better).</p>"))
        display(self.global_ranking)

        display(HTML("<h3>Table 2 — Per-Well Winners</h3><p>Best model per well by composite score (hybrid + penalty).</p>"))
        base_cols = ["well", "experiment", "score_well", "score_well_raw", "score_val", "score_test"]
        metric_cols = self.cfg.metrics
        of_cols = ["overfit_flag", "overfit_agg_flag", "overfit_cum_flag",
                   "overfit_rel_agg", "overfit_rel_cum", "overfit_abs_agg", "overfit_abs_cum"]
        cols = base_cols + metric_cols + of_cols
        display(self.per_well_winners[cols])

        # Plots
        display(HTML("<h3>Plot 1 — Overall Composite Score</h3>"))
        self._plot_global_score().show(config={"displaylogo": False, "responsive": True})

        display(HTML("<h3>Plot 2 — Winner per Well</h3>"))
        self._plot_winner_per_well().show(config={"displaylogo": False, "responsive": True})

        display(HTML("<h3>Plot 3 — Validation vs Test (with relative overfitting flags)</h3>"))
        self._plot_overfitting_scatter().show(config={"displaylogo": False, "responsive": True})

        # Executive Summary
        self._executive_summary()

    # =============================================================================
    # Plotting
    # =============================================================================
    def _plot_global_score(self) -> go.Figure:
        assert self.global_ranking is not None
        df = self.global_ranking.copy()
        df["letters"] = df["experiment"].map(self.tukey_letters).fillna("")
        fig = px.bar(
            df, x="experiment", y="mean_score", error_y="sem_score",
            color="experiment", color_discrete_map=self.style.experiment_colors,
            text=df["mean_score"].round(2),
            labels={"experiment": "Experiment", "mean_score": "Mean Composite Score (↓)"},
            template="plotly_white",
            title="<b>Overall Model Performance</b><br><span style='font-size:14px;color:#555'>Lower is better; bars show Mean ± SEM</span>",
        )
        fig.update_traces(textposition="outside", marker_line_width=1, marker_line_color="white")
        # Tukey letters
        for _, row in df.iterrows():
            y = float(row["mean_score"] + (row["sem_score"] if pd.notna(row["sem_score"]) else 0))
            fig.add_annotation(
                x=row["experiment"], y=y, yshift=12, showarrow=False,
                text=f"<b>{row['letters']}</b>",
                font=dict(size=14, family=self.style.font_family, color=self.style.color_text),
            )
        return self._apply_layout(fig)

    def _plot_winner_per_well(self) -> go.Figure:
        assert self.per_well_winners is not None
        df = self.per_well_winners.copy()
        fig = px.bar(
            df, x="well", y="score_well", color="experiment",
            color_discrete_map=self.style.experiment_colors,
            labels={"well": "Well", "score_well": "Winning Score (↓)"},
            template="plotly_white",
            title="<b>Champion per Well</b><br><span style='font-size:14px;color:#555'>Hybrid score with soft overfitting penalty</span>",
        )
        for _, row in df.iterrows():
            fig.add_annotation(
                x=row["well"], y=row["score_well"], yshift=12,
                text=f"<b>✓ {row['experiment']}</b>", showarrow=False,
                font=dict(size=12, family=self.style.font_family, color=self.style.color_text),
            )
        fig.update_layout(height=max(480, 40 * df["well"].nunique()))
        return self._apply_layout(fig)

    def _plot_overfitting_scatter(self) -> go.Figure:
        assert self.proc_df is not None
        df = self.proc_df.copy()

        # Melt and pivot to (val, test) pairs for both agg and cum
        melt = df.melt(
            id_vars=["well", "experiment", "overfit_flag"],
            value_vars=["val_smape_agg", "test_smape_agg", "val_smape_cum", "test_smape_cum"],
            var_name="metric", value_name="smape",
        )
        melt[["domain", "kind"]] = melt["metric"].str.split("_smape_", expand=True)
        pivot = melt.pivot_table(index=["well", "experiment", "kind", "overfit_flag"],
                                 columns="domain", values="smape").reset_index()

        fig = px.scatter(
            pivot, x="val", y="test", color="experiment", symbol="kind",
            color_discrete_map=self.style.experiment_colors,
            labels={"val": "Validation SMAPE", "test": "Test SMAPE"},
            template="plotly_white",
            title="<b>Validation vs Test</b><br><span style='font-size:14px;color:#555'>Points above the 45° line have higher Test error; red outline = overfitting flag (relative rule)</span>",
            hover_data=["well", "experiment", "kind"],
        )

        # Diagonal 45°
        lim = float(np.nanmax(pivot[["val", "test"]].values)) * 1.1 if not pivot.empty else 1.0
        fig.add_shape(type="line", x0=0, y0=0, x1=lim, y1=lim, line=dict(color="#999", dash="dash"))
        fig.update_xaxes(range=[0, lim]); fig.update_yaxes(range=[0, lim])

        # Emphasize flagged points by adding a scatter trace with red markers
        flagged = pivot[pivot["overfit_flag"] == True]  # noqa
        if not flagged.empty:
            fig.add_trace(go.Scatter(
                x=flagged["val"], y=flagged["test"], mode="markers",
                marker=dict(size=14, line=dict(width=2, color="#c44"), opacity=0.7),
                name="overfit flag (relative)", showlegend=True,
                hovertemplate="well=%{customdata[0]}<br>exp=%{customdata[1]}<br>kind=%{customdata[2]}<extra></extra>",
                customdata=np.stack([flagged["well"], flagged["experiment"], flagged["kind"]], axis=1),
            ))
        return self._apply_layout(fig)

    # =============================================================================
    # Reporting & Helpers
    # =============================================================================
    def _executive_summary(self) -> None:
        assert self.global_ranking is not None and self.per_well_winners is not None
        best = self.global_ranking.iloc[0]
        anova_sig, anova_p = self._anova_verdict()

        overfit_n = int(self.proc_df["overfit_flag"].sum()) if self.proc_df is not None else 0
        total_n = int(len(self.proc_df)) if self.proc_df is not None else 0
        best_wins = int(self.per_well_winners["experiment"].value_counts().get(best["experiment"], 0))
        wells_n = int(self.per_well_winners["well"].nunique())

        html = f"""
        <div style="font-family:{self.style.font_family}; color:{self.style.color_text};
                    border:1px solid #ddd; padding:14px; background:#fafafa; margin-top:18px;">
          <h3 style="margin:0 0 10px 0;">Executive Summary</h3>
          <p style="margin:0 0 12px 0; font-size:14px;">
            Protocol: <b>{self.cfg.protocol_version}</b> (Hybrid 50% rank + 50% effect) |
            Weights — Test={self.cfg.weights['test']}, Val={self.cfg.weights['val']},
            Cum={self.cfg.weights['cum']}, Agg={self.cfg.weights['agg']}
          </p>
          <ul style="line-height:1.7; margin:0;">
            <li><b>Overall winner:</b> <code>{best['experiment']}</code> with mean score <b>{best['mean_score']:.3f}</b>.</li>
            <li><b>Per-well dominance:</b> {best_wins}/{wells_n} wells won by <code>{best['experiment']}</code>.</li>
            <li><b>Significance (primary: {self.cfg.primary_metric}):</b> 
                Methods are <b>{'significantly different' if anova_sig else 'not significantly different'}</b>
                (ANOVA p={self._fmt_p(anova_p)}). Tukey letters shown on Plot 1.</li>
            <li><b>Overfitting rule (relative):</b> flag if 
                (test − val)/max(val, {self.cfg.overfit_eps} pp) ≥ {int(self.cfg.overfit_rel_tau*100)}%
                or Δ_abs ≥ {self.cfg.overfit_abs_thr} pp. 
                Soft penalty with λ={self.cfg.overfit_lambda} on excess. 
                Flags: {overfit_n}/{total_n} ({(overfit_n/total_n if total_n else 0):.1%}).</li>
          </ul>
        </div>
        """
        display(HTML(html))

    def _anova_verdict(self) -> tuple[bool, float]:
        if self.anova_table is None or "C(experiment)" not in self.anova_table.index:
            return False, float("nan")
        p = float(self.anova_table.loc["C(experiment)", "PR(>F)"])
        return (p < 0.05), p

    @staticmethod
    def _fmt_p(p: float) -> str:
        if p is None or np.isnan(p): return "n/a"
        if p < 1e-4: return "< 1e-4"
        return f"{p:.4f}"

    def _apply_layout(self, fig: go.Figure) -> go.Figure:
        fig.update_layout(
            font=dict(family=self.style.font_family, size=14, color=self.style.color_text),
            template="plotly_white",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5, title=None),
            hoverlabel=dict(bgcolor="white", font_size=13),
            margin=dict(t=90, b=70, l=30, r=20),
        )
        fig.update_yaxes(gridcolor=self.style.color_grid, zeroline=False)
        return fig

    def _find_csvs(self) -> Dict[str, Path]:
        paths: Dict[str, Path] = {}
        for exp in self.experiments:
            exact = self.reports_dir / f"{self.csv_prefix}{exp}.csv"
            if exact.exists():
                paths[exp] = exact
                continue
            # fallback: first match
            hits = sorted(self.reports_dir.glob(f"{self.csv_prefix}{exp}*.csv"))
            if hits:
                paths[exp] = hits[-1]  # most recent/last by name
        return paths

    @staticmethod
    def _compact_letter_display_from_tukey(tukey_res) -> Dict[str, str]:
        groups = sorted(set(tukey_res.groupsunique.tolist()))
        nondiff = {g: {g} for g in groups}
        rows = tukey_res._results_table.data[1:]
        for g1, g2, *_rest, reject in rows:
            if not reject:
                nondiff[g1].add(g2)
                nondiff[g2].add(g1)
        letters = {g: "" for g in groups}
        pool = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        used = 0
        remaining = set(groups)
        while remaining:
            seed = min(remaining)
            cluster = {seed}
            changed = True
            while changed:
                changed = False
                for g in list(remaining - cluster):
                    if all((g in nondiff[h]) for h in cluster):
                        cluster.add(g)
                        changed = True
            letter = pool[used % len(pool)]
            used += 1
            for g in cluster:
                letters[g] += letter
            remaining -= cluster
        return {g: "".join(sorted(set(s))) for g, s in letters.items()}


# =============================================================================
# Runner
# =============================================================================
if __name__ == "__main__":
    eval_cfg = EvaluationConfig()
    style_cfg = StyleConfig()

    suite = PIMLEvaluationSuite(
        reports_dir=REPORTS_DIR,
        experiments=EXPERIMENTS,
        csv_prefix=CSV_STEM_PREFIX,
        eval_config=eval_cfg,
        style_config=style_cfg,
    )
    suite.run()

